## Data Cleaning

#### Load data

In [76]:
import pandas as pd

listings = pd.read_csv("..\\processed\\CRMLSListing_Residential_With_Rates.csv", low_memory=False)
sold = pd.read_csv("..\\processed\\CRMLSSold_Residential_With_Rates.csv", low_memory=False)

#### Convert Data Types

Based on metadata information, convert columns to the correct intended data type.

In [77]:
def convert_dates(df):
    """
    Convert date columns to datetime format.
    
    Parameters:
    df (pd.DataFrame): The DataFrame containing the date columns.
    
    Returns:
    pd.DataFrame: The DataFrame with converted date columns.
    """
    df["CloseDate"] = pd.to_datetime(df["CloseDate"], errors="coerce")
    df["ListingContractDate"] = pd.to_datetime(df["ListingContractDate"], errors="coerce")
    df["PurchaseContractDate"] = pd.to_datetime(df["PurchaseContractDate"], errors="coerce")
    df["ContractStatusChangeDate"] = pd.to_datetime(df["ContractStatusChangeDate"], errors="coerce")
    
    return df

listings = convert_dates(listings)
sold = convert_dates(sold)

def convert_to_int64(df):
    """
    Convert columns to Int64 type.
    
    Parameters:
    df (pd.DataFrame): The DataFrame containing the columns to convert.
    
    Returns:
    pd.DataFrame: The DataFrame with converted columns.
    """
    columns_to_convert = ["YearBuilt", "BathroomsTotalInteger", "BedroomsTotal", "Stories", "MainLevelBedrooms"]
    
    for column in columns_to_convert:
        df[column] = df[column].astype('Int64', errors="ignore")
    
    return df

listings = convert_to_int64(listings)
sold = convert_to_int64(sold)


Confirm data types after conversion:

In [78]:
print("LISTINGS ---------------------------")
print(pd.DataFrame({"Dtype": listings.dtypes}))

print("\nSOLD -------------------------------")
print(pd.DataFrame({"Dtype": sold.dtypes}))

LISTINGS ---------------------------
                            Dtype
OriginalListPrice         float64
ListingKey                  int64
ListAgentEmail             object
CloseDate          datetime64[ns]
ClosePrice                float64
...                           ...
AssociationFee            float64
LotSizeSquareFeet         float64
UnparsedAddress.1          object
year_month                 object
rate_30yr_fixed           float64

[73 rows x 1 columns]

SOLD -------------------------------
                                 Dtype
Flooring                        object
ViewYN                          object
WaterfrontYN                    object
BasementYN                      object
PoolPrivateYN                   object
...                                ...
BathroomsTotalInteger_Outlier     bool
DaysOnMarket_Outlier              bool
YearBuilt_Outlier                 bool
year_month                      object
rate_30yr_fixed                float64

[82 rows x 1 columns]


#### Flag Date Inconsistencies

CloseDate must occur on or after all other dates.

In [79]:
for df in [sold, listings]:
    df["listing_after_close_flag"] = df["ListingContractDate"] > df["CloseDate"]
    df["purchase_after_close_flag"] = df["PurchaseContractDate"] > df["CloseDate"]
    df["negative_timeline_flag"] = df["ListingContractDate"] > df["PurchaseContractDate"]

    print(f"Rows with listing_after_close_flag = True: {df['listing_after_close_flag'].sum()}")
    print(f"Rows with purchase_after_close_flag = True: {df['purchase_after_close_flag'].sum()}")
    print(f"Rows with negative_timeline_flag = True: {df['negative_timeline_flag'].sum()}\n")

Rows with listing_after_close_flag = True: 74
Rows with purchase_after_close_flag = True: 243
Rows with negative_timeline_flag = True: 311

Rows with listing_after_close_flag = True: 90
Rows with purchase_after_close_flag = True: 265
Rows with negative_timeline_flag = True: 320



#### Remove Invalid Rows

Rows with invalid values for numeric datapoints will be removed: ClosePrice $\le$ 0, LivingArea $\le$ 0, DaysOnMarket < 0,
negative Bedrooms or Bathrooms.

First, we can calculate valid DaysOnMarket for invalid rows if the row has valid date consistency.

In [80]:
# Identify rows with invalid DaysOnMarket and a False listing_after_close_flag
mask = (
    (sold["listing_after_close_flag"] == False) &
    (
        sold["DaysOnMarket"].isna() |
        (sold["DaysOnMarket"] < 0)
    ) &
    sold["ListingContractDate"].notna() &
    sold["CloseDate"].notna()
)

# Calculate actual days on market
sold.loc[mask, "DaysOnMarket"] = (
    sold.loc[mask, "CloseDate"] -
    sold.loc[mask, "ListingContractDate"]
).dt.days

print(f"Recalculated DaysOnMarket for {mask.sum()} rows")

sold.loc[
    mask,
    ["ListingContractDate", "CloseDate", "DaysOnMarket",
     "listing_after_close_flag"]
].head(10)

Recalculated DaysOnMarket for 54 rows


,ListingContractDate,CloseDate,DaysOnMarket,listing_after_close_flag
4,2023-11-17,2024-01-22,66,False
393,2024-01-08,2024-01-19,11,False
11205,2024-02-24,2024-02-26,2,False
11372,2024-02-16,2024-02-23,7,False
18689,2023-12-12,2024-02-05,55,False
22996,2023-09-25,2024-02-29,157,False
24388,2024-01-30,2024-03-21,51,False
24396,2024-01-11,2024-03-19,68,False
24432,2024-03-03,2024-03-15,12,False
29416,2024-02-07,2024-03-08,30,False


Remove remaining invalid rows.

In [81]:
for name, df in [("Sold", sold), ("Listings", listings)]:
    print(f"{name}: before filtering, {len(df)} rows")

    mask = (
        (df["ClosePrice"].isna() | (df["ClosePrice"] > 0)) &
        (df["LivingArea"].isna() | (df["LivingArea"] > 0)) &
        (df["DaysOnMarket"].isna() | (df["DaysOnMarket"] >= 0)) &
        (df["BedroomsTotal"].isna() | (df["BedroomsTotal"] >= 0)) &
        (df["BathroomsTotalInteger"].isna() | (df["BathroomsTotalInteger"] >= 0))
    )

    filtered_df = df[mask].copy()

    print(f"{name}: after filtering, {len(filtered_df)} rows")

    if name == "sold":
        sold = filtered_df
    else:
        listings = filtered_df

Sold: before filtering, 480487 rows


Sold: after filtering, 480314 rows
Listings: before filtering, 644458 rows
Listings: after filtering, 644002 rows


#### Remove Columns

For our market and competitive analysis needs, we can keep only a small subset of the original columns as a separate dataset.

In [82]:
market_cols = [
    # Identifiers
    "ListingKey", "ListingKeyNumeric", "ListingId",

    # Location
    "PostalCode", "City", "CountyOrParish",
    "StateOrProvince", "MLSAreaMajor", 
    "Latitude", "Longitude", "UnparsedAddress",

    # Pricing and dates
    "ListPrice", "OriginalListPrice", "ClosePrice",
    "CloseDate", "ListingContractDate",
    "PurchaseContractDate", "ContractStatusChangeDate",
    "DaysOnMarket", "MlsStatus",

    # Property characteristics
    "PropertyType", "PropertySubType", "LivingArea",
    "BedroomsTotal", "BathroomsTotalInteger", "YearBuilt",
    "LotSizeAcres", "LotSizeArea", "LotSizeSquareFeet",
    "Stories", "Levels", "GarageSpaces", "ParkingTotal",
    "FireplaceYN", "PoolPrivateYN", "ViewYN",
    "WaterfrontYN", "BasementYN", "AttachedGarageYN",
    "NewConstructionYN", "AssociationFee",

    # Competition
    "ListAgentFullName", "ListAgentFirstName",
    "ListAgentLastName", "ListAgentAOR", "ListOfficeName",
    "BuyerAgentMlsId", "BuyerAgentFirstName",
    "BuyerAgentLastName", "BuyerAgentAOR", "BuyerOfficeName",
    "CoListAgentFirstName", "CoListAgentLastName",
    "CoListOfficeName"
]

sold_clean = sold[
    [col for col in market_cols if col in sold.columns]
].copy()

listings_clean = listings[
    [col for col in market_cols if col in listings.columns]
].copy()

print(f"Sold before vs. after shape: {sold.shape} vs. {sold_clean.shape}")
print(f"Listings before vs. after shape: {listings.shape} vs. {listings_clean.shape}")

Sold before vs. after shape: (480487, 85) vs. (480487, 54)
Listings before vs. after shape: (644002, 76) vs. (644002, 48)


Check null counts of remaining columns:

In [83]:
list_null = pd.DataFrame({
    'Dtype': listings_clean.dtypes,
    'Null Count': listings_clean.isnull().sum(),
    'Null Pct (%)': (listings_clean.isnull().sum() / len(listings_clean)) * 100
})

print("LISTINGS -------------------------------------------------------")
print(list_null)

sold_null = pd.DataFrame({
    'Dtype': sold_clean.dtypes,
    'Null Count': sold_clean.isnull().sum(),
    'Null Pct (%)': (sold_clean.isnull().sum() / len(sold_clean)) * 100
})

print("\nSOLD -----------------------------------------------------------")
print(sold_null)

LISTINGS -------------------------------------------------------
                                   Dtype  Null Count  Null Pct (%)
ListingKey                         int64           0      0.000000
ListingKeyNumeric                  int64           0      0.000000
ListingId                         object           0      0.000000
PostalCode                        object          20      0.003106
City                              object         604      0.093789
CountyOrParish                    object           0      0.000000
StateOrProvince                   object          71      0.011025
MLSAreaMajor                      object       87866     13.643746
Latitude                         float64       81183     12.606017
Longitude                        float64       81183     12.606017
UnparsedAddress                   object         968      0.150310
ListPrice                        float64           0      0.000000
OriginalListPrice                float64         931      0.1445

#### Handle Missing Values

Out of the remaining columns, WaterfrontYN and BasementYN requires handling. Null values are converted to "False" value with the assumption that data was not entered if the binary was false; all existing values are "True" values.

In [84]:
sold_clean["BasementYN"] = sold_clean["BasementYN"].fillna(False)
sold_clean["WaterfrontYN"] = sold_clean["WaterfrontYN"].fillna(False)

print(sold_clean["BasementYN"].unique())
print(sold_clean["WaterfrontYN"].unique())

[False  True]
[False  True]


C:\Users\tiffa\AppData\Local\Temp\ipykernel_14168\1196776816.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sold_clean["BasementYN"] = sold_clean["BasementYN"].fillna(False)
C:\Users\tiffa\AppData\Local\Temp\ipykernel_14168\1196776816.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sold_clean["WaterfrontYN"] = sold_clean["WaterfrontYN"].fillna(False)
